# Avito 2026 · CatBoost 起步

默认 `MODE="inspect"` 仅检查数据。显式选择 `eval` 后，使用官方 train 训练、val 选择树数；`full` 使用完整 eval 的树数在 train+val 上重训；`predict` 输出并校验提交文件。

特征和默认参数来自官方 `baseline.zip`，此 Notebook 增加相对路径、模式切换、结果管理和 Polars→Pandas 的 CatBoost 输入适配。无需附带本仓库 Python 文件；本地使用 Jigsaw，Kaggle 需提前具备 NumPy、Pandas、PyArrow、Polars、CatBoost。没有在线安装步骤。

本次初始化不执行训练；尚无自有指标或线上成绩。运行目录均为 `output/catboost/<UTC+8 timestamp>/`。完整 NDCG、@10、@30 分别报告；模型选择沿用官方 NDCG@30。

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from functools import partial
from pathlib import Path

import numpy as np
import polars as pl
from catboost import CatBoostRanker, Pool
from catboost.utils import eval_metric


In [ ]:
# inspect: metadata only; eval: train -> val; full: train+val; predict: test scores.
MODE = "inspect"
DATA_DIR = None  # Local data/ or attached Kaggle dataset; set explicitly if ambiguous.
MODEL_PATH = None  # Required for predict.
EVAL_RUN = None  # Required for full: an eval run with the selected tree count.
TRAIN_FEEDS = 0  # 0 = all feeds. A limited run is diagnostic evidence only.
VAL_FEEDS = 0
GPU_DEVICE = "0"
OUTPUT_ROOT = Path("output/catboost")


In [ ]:
def find_data_root(configured):
    if configured is not None:
        candidates = [Path(configured)]
    else:
        candidates = [Path("data"), Path("competition"), Path(".")]
        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            candidates += [p.parent for p in kaggle_input.rglob("sample_submission.parquet")]
    matches = list(dict.fromkeys(p.resolve() for p in candidates if (p / "train/impressions").is_dir()))
    if len(matches) != 1:
        raise ValueError(f"Set DATA_DIR to one extracted dataset root; found {matches}")
    return matches[0]


def new_run_directory():
    timestamp = datetime.now(timezone(timedelta(hours=8))).strftime("%Y%m%d_%H%M%S_%f")
    folder = OUTPUT_ROOT / timestamp
    folder.mkdir(parents=True, exist_ok=False)
    return folder


def write_json(path, content):
    path.write_text(json.dumps(content, indent=2, ensure_ascii=False, default=str) + "\n")


官方 CatBoost 的数值、类别与 embedding 相似度特征；不使用仅 train 可用的 `pos`。

In [ ]:
MODEL_PARAMS = {
    "boosting_type": "Plain",
    "max_ctr_complexity": 1,
    "objective": "YetiRank",
    "task_type": "GPU",
    "eval_metric": "NDCG:top=30;type=Base",
    "early_stopping_rounds": 100,
    "gpu_ram_part": 0.8,
    "learning_rate": 0.4,
    "num_trees": 2000,
    "random_state": 666,
    "l2_leaf_reg": 5,
    "nan_mode": "Min",
    "allow_writing_files": False,
    "thread_count": 8,
    "devices": "0",
}

FIT_PARAMS = {"verbose": 100, "use_best_model": True, "early_stopping_rounds": 50}

BIN_FEATURES = ["f1", "f2", "f3", "f4", "f5"]

CAT_FEATURES = ["category_id"]

FLOAT_FEATURES = [*BIN_FEATURES, "embedding_dot", "embedding_cosine", "embedding_missing"]

FEATURES = [*FLOAT_FEATURES, *CAT_FEATURES]

CANDIDATE_COLUMNS = ["item_id", *CAT_FEATURES, *BIN_FEATURES, "item_id__embedding_idx"]

EMBEDDING_FEATURES = pl.Struct(
    {
        "embedding_dot": pl.List(pl.Float32),
        "embedding_cosine": pl.List(pl.Float32),
        "embedding_missing": pl.List(pl.Float32),
    }
)


def embedding_features(batch: pl.Series, embeddings: np.memmap) -> pl.Series:
    """Compute candidate similarities once per feed; -1 denotes a missing item."""
    features = {name: [] for name in EMBEDDING_FEATURES.to_schema()}
    for row in batch:
        user = np.asarray(row["user_embedding"], dtype=np.float32)
        indices = np.asarray(row["item_id__embedding_idx"])
        missing = indices == -1
        # Convert int8 user and item vectors to float32 before multiplication.
        items = embeddings[np.maximum(indices, 0)].astype(np.float32)
        items[missing] = 0
        dot = np.einsum("ij,j->i", items, user)
        denominator = np.linalg.norm(items, axis=1) * np.linalg.norm(user)
        cosine = np.divide(dot, denominator, out=np.zeros_like(dot), where=denominator > 0)
        features["embedding_dot"].append(dot)
        features["embedding_cosine"].append(cosine)
        features["embedding_missing"].append(missing.astype(np.float32))
    result = pl.DataFrame(features, schema=EMBEDDING_FEATURES.to_schema()).to_struct()
    return result


def candidate_dataset(split: Path, max_feeds: int = 0) -> pl.LazyFrame:
    """Limit complete feeds, then expand their candidate arrays using native Polars."""
    source = pl.scan_parquet(sorted((split / "impressions").glob("*.parquet")))
    columns = CANDIDATE_COLUMNS + (["target"] if "target" in source.collect_schema() else [])
    source = source.select("feed_id", "user_embedding", *columns)
    if max_feeds:
        source = source.head(max_feeds)

    meta = json.loads((split / "item_embeddings" / "data.meta").read_text())
    embeddings = np.memmap(
        split / "item_embeddings" / "data.memmap",
        mode="r",
        dtype=meta["dtype"],
        shape=tuple(meta["shape"]),
        order=meta["order"],
    )
    candidates = (
        source.with_columns(
            pl.struct("user_embedding", "item_id__embedding_idx")
            .map_batches(
                partial(embedding_features, embeddings=embeddings),
                return_dtype=EMBEDDING_FEATURES,
                is_elementwise=True,
            )
            .alias("embedding_features")
        )
        .unnest("embedding_features")
        .explode([*columns, "embedding_dot", "embedding_cosine", "embedding_missing"])
        .drop("user_embedding", "item_id__embedding_idx")
    )
    return candidates


def read_pool(split: Path, max_feeds: int) -> Pool:
    data = candidate_dataset(split, max_feeds).collect(streaming=True).rechunk()
    pool = Pool(
        data.select(FEATURES).to_pandas(),
        label=data["target"].to_numpy(),
        cat_features=CAT_FEATURES,
        group_id=data["feed_id"].to_numpy(),
    )
    return pool


def predict_scores(batch: pl.DataFrame, model: CatBoostRanker) -> pl.DataFrame:
    features = Pool(batch.select(FEATURES).to_pandas(), cat_features=CAT_FEATURES)
    scores = model.predict(features, thread_count=MODEL_PARAMS["thread_count"])
    result = batch.select("feed_id", "item_id").with_columns(pl.Series("score", scores, dtype=pl.Float32))
    return result


In [ ]:
def ranking_metrics(frame):
    frame = frame.sort("feed_id")
    if frame.is_empty() or frame.select(pl.any_horizontal(pl.all().is_null()).any()).item():
        raise ValueError("Evaluation requires nonempty, non-null feed_id/target/score rows")
    if not frame["target"].is_in([0, 1]).all() or not frame["score"].is_finite().all():
        raise ValueError("Targets must be binary and scores finite")
    labels = frame["target"].to_list()
    scores = frame["score"].to_list()
    groups = frame["feed_id"].to_list()
    return {
        name: eval_metric(labels, scores, metric, group_id=groups, thread_count=8)[0]
        for name, metric in {
            "ndcg": "NDCG:type=Base",
            "ndcg_at_10": "NDCG:top=10;type=Base",
            "ndcg_at_30": "NDCG:top=30;type=Base",
        }.items()
    }


def validate_submission(path, data_root):
    predictions = pl.read_parquet(path)
    template = pl.read_parquet(data_root / "sample_submission.parquet")
    keys = ["feed_id", "item_id"]
    expected = {"feed_id": pl.Int32, "item_id": pl.Int32, "score": pl.Float32}
    if dict(predictions.schema) != expected:
        raise ValueError(f"Unexpected submission schema: {predictions.schema}")
    if predictions.null_count().to_numpy().any() or not predictions["score"].is_finite().all():
        raise ValueError("Submission contains null or non-finite values")
    if len(predictions) != len(template) or predictions.select(keys).n_unique() != len(predictions):
        raise ValueError("Submission has a wrong row count or duplicate keys")
    if not template.select(keys).join(predictions.select(keys), on=keys, how="anti").is_empty():
        raise ValueError("Submission keys differ from the official template")
    return {"rows": len(predictions), "feeds": predictions["feed_id"].n_unique(), "template_keys_match": True}


In [ ]:
data_root = find_data_root(DATA_DIR)
if MODE == "inspect":
    for split in ("train", "val", "test"):
        files = sorted((data_root / split / "impressions").glob("*.parquet"))
        scan = pl.scan_parquet(files)
        print(split, "parts:", len(files), "schema:", scan.collect_schema())
        print(scan.select("feed_id", "timestamp", "user_id", "item_id").head(2).collect())
elif MODE in {"eval", "full"}:
    params = {**MODEL_PARAMS, "devices": GPU_DEVICE}
    if MODE == "full" and (TRAIN_FEEDS or VAL_FEEDS):
        raise ValueError("Full training requires complete train and val")
    if MODE == "full":
        if EVAL_RUN is None:
            raise ValueError("Set EVAL_RUN to an evaluated run before full training")
        selected = json.loads((Path(EVAL_RUN) / "run.json").read_text())
        if selected["mode"] != "eval" or selected["train_feeds_limit"] or selected["val_feeds_limit"]:
            raise ValueError("Select a complete eval run, not a limited diagnostic")
        params.pop("early_stopping_rounds", None)
        params["num_trees"] = selected["trees"]
        training = pl.concat([candidate_dataset(data_root / split).collect() for split in ("train", "val")]).sort(
            "feed_id"
        )
        pool = Pool(
            training.select(FEATURES).to_pandas(),
            label=training["target"].to_numpy(),
            cat_features=CAT_FEATURES,
            group_id=training["feed_id"],
        )
        model = CatBoostRanker(**params)
        model.fit(pool, verbose=100)
        report_metrics = None
    else:
        training_pool = read_pool(data_root / "train", TRAIN_FEEDS)
        validation = candidate_dataset(data_root / "val", VAL_FEEDS).collect().sort("feed_id")
        validation_pool = Pool(
            validation.select(FEATURES).to_pandas(),
            label=validation["target"].to_numpy(),
            cat_features=CAT_FEATURES,
            group_id=validation["feed_id"].to_numpy(),
        )
        model = CatBoostRanker(**params)
        model.fit(training_pool, eval_set=validation_pool, **FIT_PARAMS)
        scores = model.predict(validation_pool, thread_count=8)
        labeled = validation.select("feed_id", "item_id", "target").with_columns(
            pl.Series("score", scores, dtype=pl.Float64)
        )
        report_metrics = ranking_metrics(labeled.select("feed_id", "target", "score"))
    run_directory = new_run_directory()
    model.save_model(str(run_directory / "model.cbm"))
    if MODE == "eval":
        labeled.write_parquet(run_directory / "val_predictions.parquet", compression="zstd")
    report = {
        "mode": MODE,
        "data_root": str(data_root),
        "trees": model.tree_count_,
        "train_feeds_limit": TRAIN_FEEDS,
        "val_feeds_limit": VAL_FEEDS,
        "model_params": model.get_all_params(),
        "metrics": report_metrics,
        "selection": "val NDCG@30" if MODE == "eval" else f"fixed trees from {EVAL_RUN}",
        "evidence": "limited diagnostic" if TRAIN_FEEDS or VAL_FEEDS else "complete local run",
    }
    write_json(run_directory / "run.json", report)
    print(run_directory, report_metrics)
elif MODE == "predict":
    if MODEL_PATH is None:
        raise ValueError("Set MODEL_PATH to a trained CatBoost checkpoint")
    model = CatBoostRanker()
    model.load_model(str(MODEL_PATH))
    candidates = candidate_dataset(data_root / "test")
    schema = candidates.select("feed_id", "item_id").collect_schema()
    schema["score"] = pl.Float32
    run_directory = new_run_directory()
    prediction_path = run_directory / "submission.parquet"
    candidates.map_batches(
        partial(predict_scores, model=model), schema=schema, projection_pushdown=False, streamable=True
    ).sink_parquet(prediction_path, compression="zstd", maintain_order=True)
    result = validate_submission(prediction_path, data_root)
    write_json(run_directory / "run.json", {"mode": MODE, "model": MODEL_PATH, "validation": result})
    print(prediction_path, result)
else:
    raise ValueError(f"Unknown MODE: {MODE}")
